# LangChain Memory: Old vs Modern Alternatives

This notebook demonstrates the legacy LangChain memory classes alongside their modern replacements.

The goal is to understand how conversational memory worked in older versions of LangChain and how the same functionality is implemented today using the new APIs.

> **Note**
>
> The memory classes shown here have been deprecated. They are included for learning purposes and for maintaining older projects. For new applications, prefer the modern alternatives based on `RunnableWithMessageHistory` or LangGraph.

---

## Memory Mapping

| Legacy Memory | Modern Alternative |
|---------------|--------------------|
| `ConversationBufferMemory` | `RunnableWithMessageHistory` |
| `ConversationBufferWindowMemory` | `RunnableWithMessageHistory` + custom history trimming |
| `ConversationSummaryMemory` | `RunnableWithMessageHistory` + conversation summarization |
| `ConversationSummaryBufferMemory` | `RunnableWithMessageHistory` + summarization + recent message window |

---

## What You'll Learn

- How each legacy memory class works
- The limitations of each approach
- How to build the same behavior using the modern LangChain APIs
---

## 🧭 Quick Navigation

1. [`ConversationBufferMemory`](#1--conversationbuffermemory-legacy) → `RunnableWithMessageHistory`
2. [`ConversationBufferWindowMemory`](#2--conversationbufferwindowmemory-legacy) → `RunnableWithMessageHistory` + trimming
3. `ConversationSummaryMemory` → `RunnableWithMessageHistory` + summarization
4. `ConversationSummaryBufferMemory` → `RunnableWithMessageHistory` + summary + recent window


# 1- ConversationBufferMemory (Legacy)

> **⚠️ Deprecated**
>
> `ConversationBufferMemory` is part of the legacy LangChain memory API. It is no longer recommended for new projects. It is included here for educational purposes and to help understand older LangChain codebases.

---

## Overview

`ConversationBufferMemory` stores the **entire conversation history** without removing or summarizing any messages.

Every time the user sends a new message, all previous user and AI messages are added to the prompt before it is sent to the language model.

```
User Message
      │
      ▼
ConversationBufferMemory
      │
      ▼
Entire Conversation History
      │
      ▼
LLM
```

---

## Example

Suppose the conversation is:

```text
User: Hi, I'm Mohamed.
AI: Nice to meet you!

User: I live in Cairo.
AI: That's great!

User: I work as an AI Engineer.
AI: Awesome!
```

Now the user asks:

```text
What do you know about me?
```

The model receives the entire conversation:

```text
Human: Hi, I'm Mohamed.
AI: Nice to meet you!

Human: I live in Cairo.
AI: That's great!

Human: I work as an AI Engineer.
AI: Awesome!

Human: What do you know about me?
```

Because every previous message is included, the model can answer correctly.

---

## Advantages

- Very easy to use.
- Never loses conversation history.
- Provides maximum context to the model.
- Great for learning how conversational memory works.

---

## Disadvantages

- Prompt size grows after every interaction.
- Higher token usage.
- Increased API cost.
- Slower responses as conversations become longer.
- Can eventually exceed the model's context window.

---

## Best Use Cases

- Short conversations.
- Learning and experimentation.
- Small chatbot projects.
- Debugging conversational applications.

---

## Key Characteristics

| Property | Value |
|----------|-------|
| Stores | Entire conversation history |
| Context Growth | Unlimited |
| Token Usage | High |
| Speed | Slower as history grows |
| Recommended for New Projects | ❌ No |

---

## Next

In the next section, we'll implement `ConversationBufferMemory` and then replace it with its modern equivalent: **`RunnableWithMessageHistory`**.

### 🔧 Setup

Configure the Groq API key and pick the model used throughout this notebook.


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "put yor api here "


model="llama-3.3-70b-versatile"

In [2]:
from langchain_groq import ChatGroq


# For normal accurate responses
llm = ChatGroq(temperature=0.0, model=model)

### Demo: `ConversationBufferMemory` in Action

Create a buffer memory object and manually add conversation turns to see how it stores **everything**, as raw message objects.


In [3]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

C:\Users\BS\AppData\Local\Temp\ipykernel_14660\3865325632.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


In [4]:
memory.save_context(
    {"input": "Hi, my name is mohamed"},  # user message
    {"output": "Hey mohamed, what's up? I'm an AI model called gim."}  # AI response
)
memory.save_context(
    {"input": "I'm researching the different types of conversational memory."},  # user message
    {"output": "That's interesting, what are some examples?"}  # AI response
)
memory.save_context(
    {"input": "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."},  # user message
    {"output": "That's interesting, what's the difference?"}  # AI response
)
memory.save_context(
    {"input": "Buffer memory just stores the entire conversation, right?"},  # user message
    {"output": "That makes sense, what about ConversationBufferWindowMemory?"}  # AI response
)
memory.save_context(
    {"input": "Buffer window memory stores the last k messages, dropping the rest."},  # user message
    {"output": "Very cool!"}  # AI response
)

**Inspecting stored messages** — `memory.chat_memory.messages` returns the full list of `HumanMessage` / `AIMessage` objects saved so far.


In [5]:
memory.chat_memory.messages

[HumanMessage(content='Hi, my name is mohamed', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hey mohamed, what's up? I'm an AI model called gim.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, wha

An alternative way to populate memory is `add_user_message()` / `add_ai_message()` directly, then `load_memory_variables({})` to see how it's rendered for injection into a prompt.


In [6]:
memory = ConversationBufferMemory(return_messages=True)

memory.chat_memory.add_user_message("Hi, my name is mohamed")
memory.chat_memory.add_ai_message("Hey mohamed, what's up? I'm an AI model called gim.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi, my name is mohamed', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey mohamed, what's up? I'm an AI model called gim.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Th

### Wiring Memory into a Chain

`ConversationChain` automatically pulls messages from `memory` and injects them before every call to the LLM.


In [7]:
from langchain_classic.chains import ConversationChain

chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

C:\Users\BS\AppData\Local\Temp\ipykernel_14660\1588115874.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(


**Test it:** ask a question that depends on earlier context, to confirm the *entire* history is available to the model.


In [8]:

chain.invoke({"input": "what is my name again?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is mohamed', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey mohamed, what's up? I'm an AI model called gim.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={})

{'input': 'what is my name again?',
 'history': [HumanMessage(content='Hi, my name is mohamed', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey mohamed, what's up? I'm an AI model called gim.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_m

In [9]:

chain.invoke({"input": "what is my name again?"})['response']



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is mohamed', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey mohamed, what's up? I'm an AI model called gim.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={})

'Your name is Mohamed, we introduced ourselves at the beginning of our conversation.'

## Modern Alternative: `RunnableWithMessageHistory`

`ConversationBufferMemory` is deprecated in recent versions of LangChain. The recommended replacement is `RunnableWithMessageHistory`, which provides the same conversational memory behavior using the LangChain Expression Language (LCEL).

Instead of attaching a memory object to a chain, `RunnableWithMessageHistory` wraps an LCEL chain and automatically loads and stores chat messages for each conversation session.

In the following example, we'll recreate the behavior of `ConversationBufferMemory` using `RunnableWithMessageHistory`.

**Step 1 — Prompt Template.** Build a chat prompt with a `MessagesPlaceholder` slot where past history will be injected.


In [10]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    ChatPromptTemplate
)

system_prompt = "You are a helpful assistant called gim."

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    MessagesPlaceholder(variable_name="history") , # here we will put the history of conversation
    HumanMessagePromptTemplate.from_template("{query}"),
])

In [11]:
# simple pipeline as always     
pipeline = prompt_template | llm

**Step 2 — Session Store.** A simple dict-based store; each `session_id` gets its own `InMemoryChatMessageHistory` — an unbounded buffer, just like the legacy memory above.


In [12]:
# this how we will inject the history to our conerstion in each new session
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map = {}
def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

**Step 3 — Wrap the Pipeline.** `RunnableWithMessageHistory` connects the pipeline to the session store, auto-loading and auto-saving messages per `session_id`.


In [13]:
from langchain_core.runnables.history import RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history, # from where to get old chats 
    input_messages_key="query",
    history_messages_key="history"
)

**Test it:** run two turns in the same session and confirm memory persists across calls.


In [14]:
pipeline_with_history.invoke(
    {"query": "my name is mohamed what about you ?"},
    config={
        "configurable": {
            "session_id": "123"
        }
    }
)

pipeline_with_history.invoke(
    {"query": "i am an ai enginner and yoy ?"},
    config={
        "configurable": {
            "session_id": "123"
        }
    }
)

AIMessage(content="That's great, Mohamed! As for me, I'm an AI assistant, which means I'm a program designed to simulate conversations, answer questions, and provide information on various topics. I don't have a personal profession like you do, but I'm here to help and assist users like you with their queries.\n\nI'm curious, what kind of AI projects do you work on, Mohamed? Are you focused on machine learning, natural language processing, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 113, 'total_tokens': 208, 'completion_time': 0.281045609, 'completion_tokens_details': None, 'prompt_time': 0.005575043, 'prompt_tokens_details': None, 'queue_time': 0.05150789, 'total_time': 0.286620652}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f38ad-7575-76c3-a874-ef62bad810fe-0'

In [15]:
pipeline_with_history.invoke(
    {"query": "what is my name ?"},
    config={
        "configurable": {
            "session_id": "123"
        }
    }
)

AIMessage(content='Your name is Mohamed.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 222, 'total_tokens': 228, 'completion_time': 0.01656012, 'completion_tokens_details': None, 'prompt_time': 0.012324383, 'prompt_tokens_details': None, 'queue_time': 0.051434597, 'total_time': 0.028884503}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f38ad-7740-79d1-a97a-40d83a92a632-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 6, 'total_tokens': 228})

# 2- ConversationBufferWindowMemory (Legacy)

> **⚠️ Deprecated**
>
> `ConversationBufferWindowMemory` is part of the legacy LangChain memory API. It is included here for educational purposes. For new projects, use `RunnableWithMessageHistory` with custom history trimming.

---

## Overview

`ConversationBufferWindowMemory` stores only the **last K conversation turns** instead of the entire conversation history.

As new messages are added, older messages are automatically removed from memory.

```
User Message
      │
      ▼
ConversationBufferWindowMemory
      │
      ▼
 Last K Messages
      │
      ▼
     LLM
```

---

## Example

Suppose:

```python
k = 2
```

Conversation:

```text
User: Hi, I'm Mohamed.
AI: Nice to meet you!

User: I live in Cairo.
AI: That's great!

User: I work as an AI Engineer.
AI: Awesome!
```

When the user asks:

```text
What do you know about me?
```

Only the last **2 conversation turns** are sent to the model:

```text
Human: I live in Cairo.
AI: That's great!

Human: I work as an AI Engineer.
AI: Awesome!

Human: What do you know about me?
```

The first conversation (`Hi, I'm Mohamed.`) is no longer included.

---

## Advantages

- Lower token usage.
- Faster than storing the full conversation.
- Prevents prompts from growing indefinitely.

---

## Disadvantages

- Older information is lost.
- The model may forget names, preferences, or earlier context.

---

## Best Use Cases

- Customer support chatbots.
- Short conversations.
- Applications where only recent context matters.

---

## Key Characteristics

| Property | Value |
|----------|-------|
| Stores | Last K conversation turns |
| Context Growth | Fixed |
| Token Usage | Low |
| Speed | Fast |
| Recommended for New Projects | ❌ No |

---

## Next

In the next section, we'll implement `ConversationBufferWindowMemory` and then recreate the same behavior using **`RunnableWithMessageHistory`** with custom history trimming.

### Demo: `ConversationBufferWindowMemory` in Action

Only the last `k=4` messages will be kept — anything older is dropped automatically.


In [16]:
from langchain_classic.memory import ConversationBufferWindowMemory 

memory = ConversationBufferWindowMemory(k=4, return_messages=True)

C:\Users\BS\AppData\Local\Temp\ipykernel_14660\1421297482.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=4, return_messages=True)


In [17]:
memory.chat_memory.add_user_message("Hi, my name is mohamed")
memory.chat_memory.add_ai_message("Hey mohamed, what's up? I'm an AI model called gim.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwa

Wire it into a `ConversationChain`, exactly like before.


In [18]:
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

**Test it:** ask about something from *outside* the window to see it get dropped.


In [19]:
chain.invoke({'input':"whats my name ?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs

{'input': 'whats my name ?',
 'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer window memory stores the last k messages, droppi

In [20]:
chain.invoke({'input':"whats my name ?"})['response']



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}), AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer window memory stores the last k messages, dropping the re

"I don't know your name. We just started talking about conversational memory, but you didn't mention your name. Would you like to share it with me?"

## Modern Alternative: `RunnableWithMessageHistory`

`ConversationBufferWindowMemory` is deprecated in recent versions of LangChain. The recommended approach is to use `RunnableWithMessageHistory` and manually control how much conversation history is passed to the model.

Instead of storing only the last **K** messages automatically, we can retrieve the full chat history and trim it to the most recent messages before each model invocation.

This provides the same behavior as `ConversationBufferWindowMemory` while giving you full control over how the conversation history is managed.

**Custom Bounded History.** There's no built-in windowed helper for `RunnableWithMessageHistory`, so we subclass `BaseChatMessageHistory` to keep only the last `k` messages.


In [21]:
from pydantic import BaseModel, Field
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

class BufferWindowMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    k: int = Field(default_factory=int)

    def __init__(self, k: int):
        super().__init__(k=k)
        print(f"Initializing BufferWindowMessageHistory with k={k}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages.
        """
        self.messages.extend(messages)
        self.messages = self.messages[-self.k:]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

**Session Factory.** Same idea as before, but each session now also takes a `k` parameter.


In [22]:
chat_map = {}
def get_chat_history2(session_id: str, k: int = 4) -> BufferWindowMessageHistory:
    print(f"get_chat_history called with session_id={session_id} and k={k}")
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = BufferWindowMessageHistory(k=k)
    # remove anything beyond the last
    return chat_map[session_id]

**Wrap the Pipeline.** `ConfigurableFieldSpec` exposes both `session_id` and `k` as runtime-configurable parameters.


In [23]:
from langchain_core.runnables import ConfigurableFieldSpec

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history2,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)

**Test it:** send several messages, then later ask about something said early on — it should have been trimmed away.


In [24]:
pipeline_with_history.invoke({'query':'hello my name is mohamed'}, config= {'session_id':'s120' , 'k':4} )

pipeline_with_history.invoke({'query':'i like ai what dou know about it ?'}, config= {'session_id':'s120' , 'k':4} )

pipeline_with_history.invoke({'query':'i live in egypt'}, config= {'session_id':'s120' , 'k':4} )

pipeline_with_history.invoke({'query':'i  graduated in cs major '}, config= {'session_id':'s120' , 'k':4} )

pipeline_with_history.invoke({'query':'i  like reading nwe info '}, config= {'session_id':'s120' , 'k':4} )


get_chat_history called with session_id=s120 and k=4
Initializing BufferWindowMessageHistory with k=4
get_chat_history called with session_id=s120 and k=4
get_chat_history called with session_id=s120 and k=4
get_chat_history called with session_id=s120 and k=4
get_chat_history called with session_id=s120 and k=4


AIMessage(content="You enjoy reading new information and staying up-to-date with the latest developments in the field of computer science and technology. That's great, Mohamed!\n\nThere are many ways to stay informed and learn new things, such as:\n\n1. **Blogs and websites**: Follow popular tech blogs and websites, such as TechCrunch, The Verge, and Hacker Noon, to stay current with the latest news and trends.\n2. **Online courses and tutorials**: Websites like Udemy, Coursera, and edX offer a wide range of courses and tutorials on various topics, including programming languages, data science, and AI.\n3. **Research papers and academic journals**: Read research papers and academic journals, such as arXiv, ResearchGate, and ACM Digital Library, to stay current with the latest research and advancements in the field.\n4. **Books and eBooks**: Read books and eBooks on topics that interest you, such as programming languages, software engineering, and computer science theory.\n5. **Podcasts

**Test with explicit instructions:** tell the model to say *"I don't know"* instead of guessing, to clearly reveal what is (and isn't) still in the trimmed window.


In [25]:
pipeline_with_history.invoke({'query':'do you know my name and where i live and like If the requested information is not present in the conversation history respond with I dont know instead of making assumptions '}, config= {'session_id':'s120' , 'k':4} ).pretty_print() 

get_chat_history called with session_id=s120 and k=4
================================== Ai Message ==================================

I don't know your name, where you live, or your personal preferences beyond what you've shared in this conversation. You mentioned earlier that you graduated with a CS major and enjoy reading new information, but that's the extent of my knowledge about you.


**Inspect stored history:** print what's actually left in the windowed buffer.


In [26]:
history = get_chat_history2("s120", 4)

for msg in history.messages:
    print(type(msg).__name__, ":", msg.content)

get_chat_history called with session_id=s120 and k=4
HumanMessage : i  like reading nwe info 
AIMessage : You enjoy reading new information and staying up-to-date with the latest developments in the field of computer science and technology. That's great, Mohamed!

There are many ways to stay informed and learn new things, such as:

1. **Blogs and websites**: Follow popular tech blogs and websites, such as TechCrunch, The Verge, and Hacker Noon, to stay current with the latest news and trends.
2. **Online courses and tutorials**: Websites like Udemy, Coursera, and edX offer a wide range of courses and tutorials on various topics, including programming languages, data science, and AI.
3. **Research papers and academic journals**: Read research papers and academic journals, such as arXiv, ResearchGate, and ACM Digital Library, to stay current with the latest research and advancements in the field.
4. **Books and eBooks**: Read books and eBooks on topics that interest you, such as programm

# ConversationSummaryMemory (Legacy)

> **⚠️ Deprecated**
>
> `ConversationSummaryMemory` is part of the legacy LangChain memory API. It is included here for educational purposes. For new projects, use `RunnableWithMessageHistory` with a conversation summarization strategy.

---

## Overview

Unlike `ConversationBufferMemory`, which stores the entire conversation, `ConversationSummaryMemory` stores a **summary** of the conversation.

As the conversation grows, the memory is continuously updated by summarizing previous interactions. This keeps the prompt small while preserving the most important information.

```
Conversation
      │
      ▼
Summarization
      │
      ▼
 Conversation Summary
      │
      ▼
      LLM
```

---

## Example

Conversation:

```text
User: Hi, I'm Mohamed.
AI: Nice to meet you!

User: I live in Egypt.
AI: Great!

User: I graduated in Computer Science.
AI: Awesome!
```

Instead of storing every message, the memory may contain:

```text
Summary:

The user is Mohamed, lives in Egypt, and graduated in Computer Science.
```

When the user asks:

```text
What do you know about me?
```

The summary is sent to the model instead of the full conversation.

---

## Advantages

- Small prompt size.
- Lower token usage.
- Suitable for long conversations.
- Prevents context window overflow.

---

## Disadvantages

- Important details may be omitted.
- Requires additional LLM calls to generate summaries.
- Summaries may become inaccurate over time.

---

## Best Use Cases

- Long-running conversations.
- AI assistants.
- Applications where efficiency is more important than preserving every message.

---

## Key Characteristics

| Property | Value |
|----------|-------|
| Stores | Conversation summary |
| Context Growth | Small |
| Token Usage | Low |
| Speed | Medium (requires summarization) |
| Recommended for New Projects | ❌ No |

---

## Next

In the next section, we'll implement `ConversationSummaryMemory` and then recreate the same behavior using **`RunnableWithMessageHistory`** with a custom summarization step.

### Demo: `ConversationSummaryMemory` in Action

This memory type uses the LLM itself to continuously compress the conversation into a running summary.


In [27]:
from langchain_classic.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(llm=llm)

C:\Users\BS\AppData\Local\Temp\ipykernel_14660\3945359647.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm)


Wire it into a `ConversationChain`.


In [28]:
chain = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=True
)

**Test it:** run a short multi-turn conversation and watch (`verbose=True`) the summary evolve after each turn.


In [29]:
chain.invoke({"input": "hello there my name is Mohamed"})
chain.invoke({"input": "I am researching the different types of conversational memory."})
chain.invoke({"input": "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory."})
chain.invoke({"input": "Buffer memory just stores the entire conversation"})
chain.invoke({"input": "Buffer window memory stores the last k messages, dropping the rest."})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: hello there my name is Mohamed
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
The human, Mohamed, introduces himself, and the AI responds with a greeting, introducing itself as a conversational AI trained on a vast amount of text data, capable of providing information on various topics, and shows interest in Mohamed's origins, mentioning the city of Cairo and its hist

{'input': 'Buffer window memory stores the last k messages, dropping the rest.',
 'history': "The human, Mohamed, introduces himself, and the AI responds with a greeting, introducing itself as a conversational AI trained on a vast amount of text data, capable of providing information on various topics, and shows interest in Mohamed's origins, mentioning the city of Cairo and its historical significance, before asking Mohamed what brings him to the conversation and what he would like to talk about. Mohamed is researching the different types of conversational memory, and the AI is delighted to chat with him about it, explaining that there are several types of conversational memory, including episodic, semantic, and procedural memory, and that conversational AI memory can be categorized into short-term and long-term memory, with the additional concept of working memory, which is essential for tasks like following a conversation and generating responses. Mohamed has been looking at Convers

Ask a follow-up question to confirm the summary preserved the key fact (the user's name).


In [30]:
chain.invoke({"input": "What is my name again?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
The human, Mohamed, introduces himself, and the AI responds with a greeting, introducing itself as a conversational AI trained on a vast amount of text data, capable of providing information on various topics, and shows interest in Mohamed's origins, mentioning the city of Cairo and its historical significance, before asking Mohamed what brings him to the conversation and what he would like to talk about. Mohamed is researching the different types of conversational memory, and the AI is delighted to chat with him about it, explaining that there are several types of conversational memory, including episodic, semantic, and procedural memory, and that conversational AI me

{'input': 'What is my name again?',
 'history': 'The human, Mohamed, introduces himself, and the AI responds with a greeting, introducing itself as a conversational AI trained on a vast amount of text data, capable of providing information on various topics, and shows interest in Mohamed\'s origins, mentioning the city of Cairo and its historical significance, before asking Mohamed what brings him to the conversation and what he would like to talk about. Mohamed is researching the different types of conversational memory, and the AI is delighted to chat with him about it, explaining that there are several types of conversational memory, including episodic, semantic, and procedural memory, and that conversational AI memory can be categorized into short-term and long-term memory, with the additional concept of working memory, which is essential for tasks like following a conversation and generating responses. Mohamed has been looking at ConversationBufferMemory and ConversationBufferWind

## Modern Alternative: `RunnableWithMessageHistory`

`ConversationSummaryMemory` is deprecated in recent versions of LangChain. The recommended approach is to use `RunnableWithMessageHistory` together with a custom summarization strategy.

Instead of storing a continuously updated summary inside a memory object, we can use `RunnableWithMessageHistory` to store the conversation history and periodically generate or update a summary using an LLM. The summary can then be injected into the prompt along with the current user message.

This approach provides the same functionality as `ConversationSummaryMemory` while offering greater flexibility and full compatibility with the modern LangChain APIs.

**Custom Summarizing History.** This class calls the LLM to rewrite the summary every time new messages arrive, then stores a single `SystemMessage` holding that summary.


In [31]:
from langchain_core.messages import SystemMessage

class ConversationSummaryMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    llm: ChatGroq = Field(default_factory=ChatGroq)

    def __init__(self, llm: ChatGroq):
        super().__init__(llm=llm)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, and summarize them.
        """
        self.messages.extend(messages)
        # construct the summary chat messages
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{messages}"
            )
        ])
        # format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=self.messages[0].content,
                messages="\n".join(x.content for x in messages)
            )
        )
        # replace the existing history with a single system summary message
        self.messages = [SystemMessage(content=new_summary.content)]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

**Session Factory** for the summarizing history — takes an `llm` in addition to the `session_id`.


In [32]:
chat_map = {}
def get_chat_history(session_id: str, llm: ChatGroq) -> ConversationSummaryMessageHistory:
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = ConversationSummaryMessageHistory(llm=llm)
    # return the chat history
    return chat_map[session_id]

**Wrap the Pipeline** and expose `llm` as a configurable field alongside `session_id`.


In [33]:
pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatGroq,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        )
    ]
)

**Test it:** start a new session and send the first message.


In [34]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is Mohamed"},
    config={"session_id": "id_123", "llm": llm}
)

AIMessage(content="Hello Mohamed, it's nice to meet you. I'm Gim, your helpful assistant. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 49, 'total_tokens': 83, 'completion_time': 0.074193323, 'completion_tokens_details': None, 'prompt_time': 0.00145104, 'prompt_tokens_details': None, 'queue_time': 0.051760938, 'total_time': 0.075644363}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f38ad-da1b-7d22-acd8-b39707c67f21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 34, 'total_tokens': 83})

**Inspect the summary** stored after just one exchange.


In [35]:
for msg in chat_map["id_123"].messages:
    print(type(msg).__name__, ":", msg.content)

SystemMessage : Here is the new conversation summary:

Mohamed introduced himself and was greeted by Gim, a helpful assistant. Gim offered assistance or a casual chat, but Mohamed has not yet responded with a specific topic or question.


Send a few more messages — the summary keeps getting regenerated in the background.


In [36]:
for msg in [
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    pipeline_with_history.invoke(
        {"query": msg},
        config={"session_id": "id_123", "llm": llm}
    )

**Test it:** confirm the earlier fact (the user's name) survived summarization.


In [37]:
pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"session_id": "id_123", "llm": llm}
)

AIMessage(content='Your name is Mohamed.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 268, 'total_tokens': 274, 'completion_time': 0.015768311, 'completion_tokens_details': None, 'prompt_time': 0.013655795, 'prompt_tokens_details': None, 'queue_time': 0.051288242, 'total_time': 0.029424106}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f38ae-1482-7733-9d81-881c4f9cb8a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 268, 'output_tokens': 6, 'total_tokens': 274})

# ConversationSummaryBufferMemory (Legacy)

> **⚠️ Deprecated**
>
> `ConversationSummaryBufferMemory` is part of the legacy LangChain memory API. It is included here for educational purposes. For new projects, use `RunnableWithMessageHistory` with a custom summarization strategy and recent message history.

---

## Overview

`ConversationSummaryBufferMemory` combines the behavior of **ConversationSummaryMemory** and **ConversationBufferMemory**.

Instead of storing either the full conversation or only a summary, it keeps:

- A **summary** of older messages.
- The **most recent messages** in their original form.

This provides a balance between preserving important context and keeping the prompt size manageable.

```
Old Messages
      │
      ▼
Conversation Summary
      │
      ├──────────────┐
      ▼              ▼
 Summary      Recent Messages
      │              │
      └──────┬───────┘
             ▼
            LLM
```

---

## Example

Conversation:

```text
User: Hi, I'm Mohamed.
AI: Nice to meet you!

User: I live in Egypt.
AI: Great!

User: I graduated in Computer Science.
AI: Awesome!

User: I enjoy reading new information.
```

The memory may contain:

```text
Summary:
The user is Mohamed, lives in Egypt, and graduated in Computer Science.

Recent Messages:
Human: I enjoy reading new information.
```

When the user asks:

```text
What do you know about me?
```

Both the summary and the recent messages are sent to the model.

---

## Advantages

- Keeps important long-term information.
- Preserves recent conversation exactly.
- More efficient than storing the full conversation.
- Better context than using a summary alone.

---

## Disadvantages

- More complex than other memory types.
- Requires periodic summarization.
- Slightly higher token usage than `ConversationSummaryMemory`.

---

## Best Use Cases

- AI assistants.
- Long-running conversations.
- Production chatbots.
- Applications requiring both long-term and recent context.

---

## Key Characteristics

| Property | Value |
|----------|-------|
| Stores | Summary + Recent Messages |
| Context Growth | Controlled |
| Token Usage | Medium |
| Speed | Medium |
| Recommended for New Projects | ❌ No |

---

## Next

In the next section, we'll implement `ConversationSummaryBufferMemory` and then recreate the same behavior using **`RunnableWithMessageHistory`** with a custom summary and message buffer.

### Demo: `ConversationSummaryBufferMemory` in Action

Set a `max_token_limit`; once exceeded, older messages get folded into a summary while recent ones stay verbatim.


In [38]:
from langchain_classic.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=300,
    return_messages=True
)

C:\Users\BS\AppData\Local\Temp\ipykernel_14660\114919256.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(


Wire it into a `ConversationChain`.


In [39]:
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

**Test it:** run a multi-turn conversation and watch the summary + recent-message split happen (`verbose=True`).


In [40]:
for msg in [
    "hi my name is mohamed ",
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    chain.invoke({"input": msg})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: hi my name is mohamed 
AI:


c:\Users\BS\anaconda3\envs\py_12\Lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))



> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='hi my name is mohamed ', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Mohamed, it's lovely to meet you. I'm an AI designed to assist and communicate with humans in a helpful and informative way. I have been trained on a vast amount of text data, which allows me to provide detailed answers to a wide range of questions and topics. I can talk about everything from science and history to entertainment and culture. I'm excited to chat with you, Mohamed, and learn more about your interests. By the way, I've been trained on data up to 2023, so I'm aware of events, discoveries, and trends up to that point. What

Ask a follow-up question to confirm both the summary and the buffer preserved enough context.


In [41]:

chain.invoke({"input": "what is my name ?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[SystemMessage(content='The human, Mohamed, introduces himself and the AI responds with a greeting and an introduction to its capabilities. The AI explains that it can provide detailed answers to a wide range of questions and topics, and mentions its training data is up to 2023. Mohamed expresses interest in researching conversational memory, and the AI explains the different types, including episodic, semantic, and working memory. The AI further categorizes conversational memory in the context of conversational AI into short-term and long-term memory, and mentions the use of contextual memory to understand the conversation context and adapt responses. The AI also disc

{'input': 'what is my name ?',
 'history': [SystemMessage(content='The human, Mohamed, introduces himself and the AI responds with a greeting and an introduction to its capabilities. The AI explains that it can provide detailed answers to a wide range of questions and topics, and mentions its training data is up to 2023. Mohamed expresses interest in researching conversational memory, and the AI explains the different types, including episodic, semantic, and working memory. The AI further categorizes conversational memory in the context of conversational AI into short-term and long-term memory, and mentions the use of contextual memory to understand the conversation context and adapt responses. The AI also discusses its training on various conversational memory models, including the Transformer architecture, and offers to provide more information on specific aspects of conversational memory that Mohamed is interested in. Mohamed inquires about ConversationBufferMemory and ConversationB

## Modern Alternative: `RunnableWithMessageHistory`

`ConversationSummaryBufferMemory` is deprecated in recent versions of LangChain. The recommended approach is to use `RunnableWithMessageHistory` with a custom summarization strategy and a recent message buffer.

Instead of relying on a built-in memory class, we can store a running summary of older conversations while keeping the most recent messages unchanged. During each request, both the summary and the recent messages are injected into the prompt.

This approach provides the same behavior as `ConversationSummaryBufferMemory` while offering greater flexibility and full compatibility with the modern LangChain APIs.

**Custom Summary + Buffer History.** Combines the previous two techniques: keep the last `k` messages verbatim, and fold anything older into a single running `SystemMessage` summary.


In [42]:
class ConversationSummaryBufferMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    llm: ChatGroq = Field(default_factory=ChatGroq)
    k: int = Field(default_factory=int)

    def __init__(self, llm: ChatGroq, k: int):
        super().__init__(llm=llm, k=k)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages and summarizing the messages that we
        drop.
        """
        existing_summary: SystemMessage | None = None
        old_messages: list[BaseMessage] | None = None
        # see if we already have a summary message
        if len(self.messages) > 0 and isinstance(self.messages[0], SystemMessage):
            print(">> Found existing summary")
            existing_summary = self.messages.pop(0)
        # add the new messages to the history
        self.messages.extend(messages)
        # check if we have too many messages
        if len(self.messages) > self.k:
            print(
                f">> Found {len(self.messages)} messages, dropping "
                f"oldest {len(self.messages) - self.k} messages.")
            # pull out the oldest messages...
            old_messages = self.messages[:-self.k]
            # ...and keep only the most recent messages
            self.messages = self.messages[-self.k:]
        if old_messages is None:
            print(">> No old messages to update summary with")
            # if we have no old_messages, we have nothing to update in summary
            return
        # construct the summary chat messages
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{old_messages}"
            )
        ])
        # format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary,
                old_messages=old_messages
            )
        )
        print(f">> New summary: {new_summary.content}")
        # prepend the new summary to the history
        self.messages = [SystemMessage(content=new_summary.content)] + self.messages

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

**Session Factory** — now takes `llm` and `k` together.


In [43]:
chat_map = {}
def get_chat_history(session_id: str, llm: ChatGroq, k: int) -> ConversationSummaryBufferMessageHistory:
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = ConversationSummaryBufferMessageHistory(llm=llm, k=k)
    # return the chat history
    return chat_map[session_id]

**Wrap the Pipeline**, exposing `session_id`, `llm`, and `k` as configurable fields.


In [44]:
pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatGroq,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)

**Test it:** send a sequence of messages and watch the console output show summarization kick in once the window overflows.


In [45]:
for i, msg in enumerate([
    "hello my name is mohamed",
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]):
    print(f"---\nMessage {i+1}\n---\n")
    pipeline_with_history.invoke(
        {"query": msg},
        config={"session_id": "id_123", "llm": llm, "k": 4}
    )

---
Message 1
---

>> No old messages to update summary with
---
Message 2
---

>> No old messages to update summary with
---
Message 3
---

>> Found 6 messages, dropping oldest 2 messages.
>> New summary: Here is a new summary of the conversation:

The conversation started with a greeting from a user named Mohamed. The assistant, Gim, responded with a friendly hello and introduced itself as a helpful assistant. Gim asked Mohamed if there was something specific they needed help with or if they would like to chat. The conversation has just begun, and no specific topic or issue has been discussed yet.
---
Message 4
---

>> Found existing summary
>> Found 6 messages, dropping oldest 2 messages.
>> New summary: Here is a new summary of the conversation:

The conversation started with a greeting from a user named Mohamed. The assistant, Gim, responded with a friendly hello and introduced itself as a helpful assistant. Gim asked Mohamed if there was something specific they needed help with o